# AAIT (III–II) – EXP-12  
## Developing a Simple Chatbot using Python (VS Code / Google Colab)

**Aim:** Design and implement a simple text-based chatbot in Python that demonstrates:
- Basic **intent design**
- **Rule-based** response selection using keywords/phrases
- Continuous conversation using a **loop**

**Tools:** Python (VS Code / Google Colab)  
**Mapped CO:** CO5

---
### What you will build (End result)
A small **AAIT Helpdesk Bot** that can answer a limited set of questions like:
- Greetings
- AAIT tools info (Trello/Figma/VP/Colab/Teachable Machine)
- Lab-related FAQs (generic)
- Submission / evaluation type questions (generic)

> Note: This is a **rule-based** chatbot (not AI/ML). The “intelligence” comes from good **intents + patterns + responses**.


## Quick idea: How a rule-based chatbot works

1. User types a message  
2. We **preprocess** the text (lowercase, remove extra spaces, etc.)  
3. We check which **intent** matches best using keyword/phrase rules  
4. We print the mapped response  
5. Repeat until user types `exit` / `bye`

**Intent examples**
- greeting → “hi”, “hello”
- tools_info → “trello”, “figma”, “teachable machine”
- lab_info → “lab”, “experiment”, “submission”


In [2]:

# EXP-12: Rule-Based Chatbot (AAIT Helpdesk Bot)
# -------------------------------------------------
# This notebook is ready-to-run. There are NO blanks to fill.
# Students can improve it by adding more intents/patterns/responses.

import random
import re
from datetime import datetime

## Step 1 — Define intents, patterns, and responses

We store chatbot knowledge in a single dictionary:
- **patterns**: keywords/phrases that indicate an intent
- **responses**: possible replies for that intent

Tip: Keep patterns realistic (words students will actually type).


In [3]:

BOT_NAME = "AAIT Helpdesk Bot"

# Exit commands (type any of these to stop)
EXIT_COMMANDS = {"exit", "quit", "bye", "stop"}

# Intent knowledge base
# Keep responses generic (safe) because rules/policies vary by section/institute.
INTENTS = {
    "greeting": {
        "patterns": ["hi", "hello", "hey", "good morning", "good afternoon", "good evening"],
        "responses": [
            "Hello! I am the AAIT Helpdesk Bot. How can I help you?",
            "Hi! Ask me about AAIT tools, lab experiments, or submissions."
        ]
    },
    "aait_course": {
        "patterns": ["what is aait", "aait", "application of ai tools", "ai tools course"],
        "responses": [
            "AAIT stands for Application of AI Tools. In this lab you learn tools like Trello, Visual Paradigm, Figma, Teachable Machine, Colab, and Tableau.",
            "AAIT is a hands-on lab course where you plan, design, and build small AI prototypes using modern tools."
        ]
    },
    "tools_info": {
        "patterns": ["trello", "figma", "visual paradigm", "vp", "teachable machine", "colab", "tableau", "github"],
        "responses": [
            "AAIT tools commonly include Trello (planning), Visual Paradigm (flowcharts), Figma (UI wireframes), Teachable Machine/Colab (models), and Tableau (visualization).",
            "If you tell me the tool name (Trello/Figma/VP/Colab/Teachable Machine), I can explain what it is used for in AAIT."
        ]
    },
    "lab_info": {
        "patterns": ["lab", "experiment", "practical", "record", "procedure"],
        "responses": [
            "In AAIT lab, focus on: understanding the task, following steps, generating your own output, and writing reflection in your own words.",
            "For lab work: always keep your outputs ready, and be prepared to explain your steps and results."
        ]
    },
    "submission": {
        "patterns": ["submit", "submission", "google form", "deadline", "pdf", "link", "upload"],
        "responses": [
            "Submission is usually through Google Form / link upload. Ensure your file/link is accessible and your output is clearly visible.",
            "Before submitting: re-run your notebook/script fully and verify outputs are generated by you."
        ]
    },
    "evaluation": {
        "patterns": ["marks", "evaluation", "viva", "grading", "exam", "comprehensive"],
        "responses": [
            "AAIT evaluation usually checks: correct workflow, outputs, explanation, and reflection. Be ready for viva questions on your own work.",
            "For exams/viva: know your algorithm steps, flowchart meaning, and why your tool/model choice fits the problem."
        ]
    },
    "time_now": {
        "patterns": ["time now", "current time", "what time", "time please"],
        "responses": [
            # We will handle this intent dynamically in code (see get_response()).
            "DYNAMIC_TIME"
        ]
    },
    "thanks": {
        "patterns": ["thank you", "thanks", "thx"],
        "responses": [
            "You're welcome!",
            "Glad to help. Ask me anytime."
        ]
    },
    "goodbye": {
        "patterns": ["bye", "goodbye", "see you"],
        "responses": [
            "Goodbye. All the best for your AAIT lab!",
            "See you. Keep practicing and keep your outputs ready."
        ]
    }
}

# Default responses if nothing matches
UNKNOWN_RESPONSES = [
    "Sorry, I didn’t understand that. Try asking about AAIT, tools, lab, submission, or evaluation.",
    "I’m not sure about that. Ask me about AAIT tools or lab submissions."
]

## Step 2 — Preprocess user input

We do simple cleanup to make matching easier:
- lowercase
- remove extra spaces
- remove punctuation (basic)


In [4]:

def preprocess(text: str) -> str:
    """Lowercase, remove punctuation (basic), and normalize spaces."""
    text = text.lower().strip()
    # Replace punctuation with spaces (so words don't join together)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    # Normalize multiple spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

## Step 3 — Match intent (better than “first match”)

Instead of returning the first match, we:
- calculate a **score** for each intent (how many patterns match)
- choose the intent with the **highest score**

This gives more stable results when a message contains multiple keywords.


In [5]:

def match_intent(user_text: str) -> str:
    """Return the best-matching intent using simple scoring."""
    text = preprocess(user_text)
    tokens = set(text.split())

    best_intent = None
    best_score = 0

    for intent, data in INTENTS.items():
        score = 0
        for pat in data["patterns"]:
            pat_clean = preprocess(pat)

            # If pattern is a phrase (contains space), use substring match
            if " " in pat_clean:
                if pat_clean in text:
                    score += 2  # phrase match gets slightly higher weight
            else:
                # single word: match as token
                if pat_clean in tokens:
                    score += 1

        if score > best_score:
            best_score = score
            best_intent = intent

    return best_intent if best_score > 0 else "unknown"

## Step 4 — Generate response

Some intents can be **dynamic** (example: current time).
We also keep a small `context` dictionary to store basic info (optional).


In [6]:

def get_response(user_text: str, context: dict) -> str:
    intent = match_intent(user_text)
    context["last_intent"] = intent

    if intent == "time_now":
        now = datetime.now().strftime("%I:%M %p")
        return f"Current time is {now}."

    if intent == "unknown":
        return random.choice(UNKNOWN_RESPONSES)

    return random.choice(INTENTS[intent]["responses"])

## Step 5 — Continuous chat loop (until exit)

- Reads user input
- Checks exit command
- Prints a mapped response


In [7]:

def chat():
    print(f"{BOT_NAME}: Hi! Type 'exit' to quit.")
    context = {"last_intent": None}

    while True:
        user_input = input("You: ").strip()
        if not user_input:
            print(f"{BOT_NAME}: Please type something (or type 'exit' to quit).")
            continue

        cleaned = preprocess(user_input)
        if cleaned in EXIT_COMMANDS:
            print(f"{BOT_NAME}: Goodbye.")
            break

        reply = get_response(user_input, context)
        print(f"{BOT_NAME}: {reply}")

## Quick Demo (non-interactive)

Run this to see sample outputs in the notebook before starting live chat.


In [8]:

def demo():
    sample_inputs = [
        "Hi",
        "What is AAIT?",
        "Tell me about Trello and Figma",
        "How to submit the experiment?",
        "What about marks and viva?",
        "What time now?",
        "random unknown question"
    ]
    context = {"last_intent": None}

    for s in sample_inputs:
        print("You:", s)
        print(f"{BOT_NAME}:", get_response(s, context))
        print("-")

demo()

You: Hi
AAIT Helpdesk Bot: Hi! Ask me about AAIT tools, lab experiments, or submissions.
-
You: What is AAIT?
AAIT Helpdesk Bot: AAIT stands for Application of AI Tools. In this lab you learn tools like Trello, Visual Paradigm, Figma, Teachable Machine, Colab, and Tableau.
-
You: Tell me about Trello and Figma
AAIT Helpdesk Bot: AAIT tools commonly include Trello (planning), Visual Paradigm (flowcharts), Figma (UI wireframes), Teachable Machine/Colab (models), and Tableau (visualization).
-
You: How to submit the experiment?
AAIT Helpdesk Bot: In AAIT lab, focus on: understanding the task, following steps, generating your own output, and writing reflection in your own words.
-
You: What about marks and viva?
AAIT Helpdesk Bot: AAIT evaluation usually checks: correct workflow, outputs, explanation, and reflection. Be ready for viva questions on your own work.
-
You: What time now?
AAIT Helpdesk Bot: Current time is 01:50 PM.
-
You: random unknown question
AAIT Helpdesk Bot: I’m not sure

## Start the chatbot (interactive)

Run the next cell and chat in a loop.  
Type `exit` / `bye` to stop.


In [9]:
# Run the chatbot (interactive)
# -----------------------------
# Execute this cell and start chatting.
# Type: exit / bye / quit  to stop.

chat()

AAIT Helpdesk Bot: Hi! Type 'exit' to quit.
You: Exit
AAIT Helpdesk Bot: Goodbye.


## How to improve this chatbot (ideas for students)

Try any 3–5 improvements:
1. **Add new intents** (attendance rules, lab location, experiment list, etc.)  
2. Add more **patterns** (real student language like “sir lab eppudu”, “timings cheppandi”)  
3. Add **better matching** (fuzzy match using `difflib`, or regex word boundaries)  
4. Add **conversation memory** (example: if last intent was tools_info, handle follow-up like “and what about VP?”)  
5. Store a **small FAQ database** in a CSV and read it into the bot (still rule-based)  
6. (Advanced) Convert to a basic ML intent classifier using TF-IDF + Logistic Regression (future upgrade)

**Reflection tip:** Mention how your chatbot behaves for valid queries vs unknown queries, and what rules you used.


# Task
The task is to enhance the existing rule-based chatbot by:
1.  **Expanding Intents and Patterns**: Add new intents like 'attendance_rules', 'lab_location', and 'experiment_list' to the `INTENTS` dictionary in cell `f9000ed0`, and include more diverse and realistic student language patterns (e.g., “sir lab eppudu”, “timings cheppandi”) for both existing and new intents.
2.  **Improving Matching Logic**: Modify the `preprocess` function in cell `20ffbffa` and the `match_intent` function in cell `be2b541c` to utilize regular expressions with word boundaries (`\b`) for more accurate and precise pattern matching.
3.  **Updating Unknown Responses**: Update the `UNKNOWN_RESPONSES` list in cell `f9000ed0` to reflect the expanded capabilities of the bot.
4.  **Testing**: Add new test cases to the `sample_inputs` list within the `demo` function in cell `76243f24` to thoroughly validate the new intents, patterns, and regex-based matching.
5.  **Summarize Enhancements**: Provide a summary of the improvements, highlighting the new intents, expanded vocabulary, and the more robust matching mechanism, and explain how these contribute to a more intelligent and user-friendly chatbot.

## Expand Intents and Patterns

### Subtask:
Modify the `INTENTS` dictionary in the `f9000ed0` cell to include new intents such as 'attendance_rules', 'lab_location', and 'experiment_list'. For both existing and new intents, add more diverse patterns, including examples of 'real student language' like “sir lab eppudu” and “timings cheppandi”.


**Reasoning**:
The user wants to modify the `INTENTS` dictionary in cell `f9000ed0` by adding new intents and expanding patterns for existing ones. I will provide the complete updated code for that cell.



In [10]:
BOT_NAME = "AAIT Helpdesk Bot"

# Exit commands (type any of these to stop)
EXIT_COMMANDS = {"exit", "quit", "bye", "stop", "goodbye"}

# Intent knowledge base
# Keep responses generic (safe) because rules/policies vary by section/institute.
INTENTS = {
    "greeting": {
        "patterns": [
            "hi", "hello", "hey", "good morning", "good afternoon", "good evening",
            "hi bot", "hello there", "hey bot", "namaste", "howdy"
        ],
        "responses": [
            "Hello! I am the AAIT Helpdesk Bot. How can I help you?",
            "Hi! Ask me about AAIT tools, lab experiments, or submissions.",
            "Hey there! What can I assist you with today?"
        ]
    },
    "aait_course": {
        "patterns": [
            "what is aait", "aait", "application of ai tools", "ai tools course",
            "tell me about aait", "aait subject", "what is this aait lab"
        ],
        "responses": [
            "AAIT stands for Application of AI Tools. In this lab you learn tools like Trello, Visual Paradigm, Figma, Teachable Machine, Colab, and Tableau.",
            "AAIT is a hands-on lab course where you plan, design, and build small AI prototypes using modern tools.",
            "It's a practical course to learn various AI-related tools for development and analysis."
        ]
    },
    "tools_info": {
        "patterns": [
            "trello", "figma", "visual paradigm", "vp", "teachable machine", "colab", "tableau", "github",
            "about trello", "what is figma for", "use of vp", "colab uses", "teachable machine purpose",
            "what tools are used in aait", "tools for aait"
        ],
        "responses": [
            "AAIT tools commonly include Trello (planning), Visual Paradigm (flowcharts), Figma (UI wireframes), Teachable Machine/Colab (models), and Tableau (visualization).",
            "If you tell me the tool name (Trello/Figma/VP/Colab/Teachable Machine), I can explain what it is used for in AAIT.",
            "We use tools like Trello for project management, Figma for design, and Colab for coding and model training."
        ]
    },
    "lab_info": {
        "patterns": [
            "lab", "experiment", "practical", "record", "procedure", "lab work",
            "sir lab eppudu", "lab timings", "timings cheppandi", "about lab", "lab procedures",
            "how to do lab", "what to do in lab"
        ],
        "responses": [
            "In AAIT lab, focus on: understanding the task, following steps, generating your own output, and writing reflection in your own words.",
            "For lab work: always keep your outputs ready, and be prepared to explain your steps and results.",
            "Make sure to follow the experiment instructions carefully and document your findings."
        ]
    },
    "submission": {
        "patterns": [
            "submit", "submission", "google form", "deadline", "pdf", "link", "upload",
            "how to submit", "where to submit", "submission link", "submission process",
            "when is submission", "last date for submission", "how to upload"
        ],
        "responses": [
            "Submission is usually through Google Form / link upload. Ensure your file/link is accessible and your output is clearly visible.",
            "Before submitting: re-run your notebook/script fully and verify outputs are generated by you.",
            "Always check the submission guidelines and deadlines provided by your instructor."
        ]
    },
    "evaluation": {
        "patterns": [
            "marks", "evaluation", "viva", "grading", "exam", "comprehensive",
            "how is evaluation done", "viva questions", "how are marks given", "exam format"
        ],
        "responses": [
            "AAIT evaluation usually checks: correct workflow, outputs, explanation, and reflection. Be ready for viva questions on your own work.",
            "For exams/viva: know your algorithm steps, flowchart meaning, and why your tool/model choice fits the problem.",
            "The evaluation focuses on your understanding and practical application of the concepts."
        ]
    },
    "attendance_rules": {
        "patterns": [
            "attendance rules", "is attendance compulsory", "attendance percentage", "sir attendance entha",
            "attendance policy", "minimum attendance", "do we need attendance"
        ],
        "responses": [
            "Please refer to your course syllabus or contact your instructor/department for the specific attendance rules and requirements.",
            "Attendance policies can vary; always check the official guidelines for minimum percentage and consequences."
        ]
    },
    "lab_location": {
        "patterns": [
            "lab location", "where is the lab", "lab room number", "lab ekkada",
            "what is the lab room", "which lab is aait"
        ],
        "responses": [
            "The AAIT lab location and room number are usually communicated by your faculty. Please check your timetable or class announcements.",
            "I don't have real-time lab location data. Please ask your instructor or check the campus map/schedule."
        ]
    },
    "experiment_list": {
        "patterns": [
            "experiment list", "what experiments", "exp list", "experiments cheppandi",
            "list of experiments", "aait experiment names", "how many experiments"
        ],
        "responses": [
            "The list of experiments for AAIT will be provided by your instructor at the beginning of the semester or as each experiment is assigned.",
            "Typically, you will cover experiments related to planning (Trello), design (Figma, VP), model building (Teachable Machine, Colab), and visualization (Tableau)."
        ]
    },
    "time_now": {
        "patterns": ["time now", "current time", "what time", "time please", "whats the time"],
        "responses": [
            # We will handle this intent dynamically in code (see get_response()).
            "DYNAMIC_TIME"
        ]
    },
    "thanks": {
        "patterns": ["thank you", "thanks", "thx", "tnx", "thank you bot", "appreciate it"],
        "responses": [
            "You're welcome!",
            "Glad to help. Ask me anytime.",
            "No problem!"
        ]
    },
    "goodbye": {
        "patterns": ["bye", "goodbye", "see you", "cya", "farewell", "talk to you later"],
        "responses": [
            "Goodbye. All the best for your AAIT lab!",
            "See you. Keep practicing and keep your outputs ready.",
            "Farewell!"
        ]
    }
}

# Default responses if nothing matches
UNKNOWN_RESPONSES = [
    "Sorry, I didn’t understand that. Try asking about AAIT, tools, lab, submission, or evaluation.",
    "I’m not sure about that. Ask me about AAIT tools or lab submissions.",
    "Could you please rephrase that? I'm designed to help with AAIT lab-related queries."
]

## Implement Improved Matching with Regex Word Boundaries

### Subtask:
Update the `preprocess` function (cell `20ff0ffa`) and the `match_intent` function (cell `be2b541c`). Modify the intent matching logic to use regular expressions with word boundaries (`\b`) for pattern checking. This will ensure that keywords and phrases are matched precisely as whole words, rather than as substrings, improving the accuracy of intent detection.


**Reasoning**:
As instructed, I will first confirm that the `preprocess` function in cell `20ffbffa` is suitable for word boundary matching, which it is. Then, I will provide the updated `match_intent` function for cell `be2b541c` to implement the regular expression-based pattern matching with word boundaries, as specified in the instructions. This involves replacing the existing matching logic with `re.search()` using patterns constructed with `\b` and `re.escape()`.



In [11]:
def match_intent(user_text: str) -> str:
    """Return the best-matching intent using simple scoring with regex word boundaries."""
    text = preprocess(user_text)

    best_intent = None
    best_score = 0

    for intent, data in INTENTS.items():
        score = 0
        for pat in data["patterns"]:
            pat_clean = preprocess(pat)

            # Create a regex pattern with word boundaries for precise matching
            # re.escape() is used to treat special regex characters in pat_clean literally
            regex_pattern = r'\b' + re.escape(pat_clean) + r'\b'

            # Use re.search to find if the pattern exists as a whole word/phrase in the text
            if re.search(regex_pattern, text):
                score += 1  # Increment score for each matching pattern

        if score > best_score:
            best_score = score
            best_intent = intent

    return best_intent if best_score > 0 else "unknown"